In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


RU_MONTHS = {
    1: "январь",
    2: "февраль",
    3: "март",
    4: "апрель",
    5: "май",
    6: "июнь",
    7: "июль",
    8: "август",
    9: "сентябрь",
    10: "октябрь",
    11: "ноябрь",
    12: "декабрь",
}


In [2]:
def expand_date_range(start, end):
    if pd.isna(start) or pd.isna(end):
        return []
    if end < start:
        return []
    return pd.date_range(start=start, end=end, freq="D").date.tolist()


def transform_initial_plan(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    text_cols = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    if "Дата.приема.2026" in df.columns:
        df["Дата.приема.2026"] = pd.to_datetime(
            df["Дата.приема.2026"], errors="coerce"
        ).dt.date

    if "Дата.увольнения.2026" in df.columns:
        df["Дата.увольнения.2026"] = pd.to_datetime(
            df["Дата.увольнения.2026"], errors="coerce"
        ).dt.date

    df["Дата"] = df.apply(
        lambda row: expand_date_range(
            row.get("Дата.приема.2026"),
            row.get("Дата.увольнения.2026"),
        ),
        axis=1,
    )

    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    desired_order = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
        "Дата.приема.2026",
        "Дата.увольнения.2026",
        "Дата",
        "Примечание",
    ]
    existing_order = [col for col in desired_order if col in df.columns]
    other_cols = [col for col in df.columns if col not in existing_order]
    df = df[existing_order + other_cols]

    df = df.rename(
        columns={
            "Дата.приема.2026": "Дата начала работы",
            "Дата.увольнения.2026": "Дата окончания работы",
            "Источник": "План/Факт",
            "ФИО ": "ФИО",
            "Должность": "Профессия",
        }
    )

    df = df[pd.to_datetime(df["Дата"], errors="coerce").dt.year == 2026].copy()
    return df


def transform_staff_count(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    text_cols = [
        "План/Факт",
        "Подразделение",
        "Отдел",
        "ФИО работника",
        "Должность/профессияработника",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    if "Дата начала работы" in df.columns:
        df["Дата начала работы"] = pd.to_datetime(
            df["Дата начала работы"], errors="coerce"
        ).dt.date

    if "Дата окончания работы" in df.columns:
        df["Дата окончания работы"] = pd.to_datetime(
            df["Дата окончания работы"], errors="coerce"
        ).dt.date

    df["Дата"] = df.apply(
        lambda row: expand_date_range(
            row.get("Дата начала работы"),
            row.get("Дата окончания работы"),
        ),
        axis=1,
    )

    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    df.drop(columns=["Год", "Пол", "Основная профессия при приеме"], errors="ignore", inplace=True)

    desired_order = [
        "Подразделение",
        "План/Факт",
        "Отдел",
        "ФИО работника",
        "Должность/профессияработника",
        "Дата начала работы",
        "Дата окончания работы",
        "Дата",
    ]
    existing_order = [col for col in desired_order if col in df.columns]
    other_cols = [col for col in df.columns if col not in existing_order]
    df = df[existing_order + other_cols]

    df = df.rename(
        columns={
            "Отдел": "ЦФО",
            "ФИО работника": "ФИО",
            "Должность/профессияработника": "Профессия",
        }
    )
    return df


In [18]:
def _to_datetime_safe(df: pd.DataFrame, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def add_resource_shift_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет служебные столбцы для строк
    'Ресурсы для выполнения программы':
    - Ресурсы_t
    - Ресурсы_t+1
    - Ресурсы_t+7
    - Дата_t+1
    - Дата_t+7
    - ДеньМесяц_t+1
    - ДеньМесяц_t+7

    Логика:
    строка с датой T и План/Факт = 'Ресурсы для выполнения программы'
    будет учитываться:
    - в T как Ресурсы_t
    - в T-1 как Ресурсы_t+1
    - в T-7 как Ресурсы_t+7
    """
    df = df.copy()

    is_resource = df["План/Факт"].eq("Ресурсы для выполнения программы")


    df["Ресурсы_t"] = np.where(is_resource, 1, 0).astype("int8")
    df["Ресурсы_t+1"] = np.where(is_resource, 1, 0).astype("int8")
    df["Ресурсы_t+7"] = np.where(is_resource, 1, 0).astype("int8")

    df["Дата_t+1"] = pd.NaT
    df["Дата_t+7"] = pd.NaT

    df.loc[is_resource, "Дата_t+1"] = df.loc[is_resource, "Дата"] - pd.Timedelta(days=1)
    df.loc[is_resource, "Дата_t+7"] = df.loc[is_resource, "Дата"] - pd.Timedelta(days=7)

    df["ДеньМесяц_t+1"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df["ДеньМесяц_t+7"] = pd.Series(pd.NA, index=df.index, dtype="string")

    df.loc[is_resource, "ДеньМесяц_t+1"] = (
        pd.to_datetime(df.loc[is_resource, "Дата_t+1"], errors="coerce")
        .dt.strftime("%d.%m")
        .astype("string")
    )
    df.loc[is_resource, "ДеньМесяц_t+7"] = (
        pd.to_datetime(df.loc[is_resource, "Дата_t+7"], errors="coerce")
        .dt.strftime("%d.%m")
        .astype("string")
    )

    return df


def transform_registry(
    df_initial_plan: pd.DataFrame,
    df_plan_count: pd.DataFrame,
    df_fact_count: pd.DataFrame,
) -> pd.DataFrame:
    df = pd.concat(
        [df_initial_plan, df_plan_count, df_fact_count],
        ignore_index=True,
        sort=False,
    ).copy()

    df = _to_datetime_safe(df, ["Дата", "Дата начала работы", "Дата окончания работы"])

    df["_ГодФильтр"] = df["Дата"].dt.year
    df = df[df["_ГодФильтр"].isin([2025, 2026])].copy()

    df.drop(columns=["Примечание", "_ГодФильтр"], errors="ignore", inplace=True)

    df["Год"] = df["Дата"].dt.year

    if "Подразделение" in df.columns:
        df["Подразделение"] = df["Подразделение"].astype("string")
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Административно-хозяйственный отдел",
            "Администрация",
            regex=False,
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Участок ",
            "",
            regex=False,
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Эрел",
            "Хаптагай-Хая",
            regex=False,
        )

    df["ДеньМесяц"] = df["Дата"].dt.strftime("%d.%m")
    df["Месяц"] = df["Дата"].dt.month.map(RU_MONTHS)

    fio = df["ФИО"].astype("string").fillna("") if "ФИО" in df.columns else pd.Series("", index=df.index)
    подразделение = df["Подразделение"].astype("string").fillna("") if "Подразделение" in df.columns else pd.Series("", index=df.index)
    профессия = df["Профессия"].astype("string").fillna("") if "Профессия" in df.columns else pd.Series("", index=df.index)

    дата_начала = (
        pd.to_datetime(df["Дата начала работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата начала работы" in df.columns else pd.Series("", index=df.index)
    )
    дата_окончания = (
        pd.to_datetime(df["Дата окончания работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата окончания работы" in df.columns else pd.Series("", index=df.index)
    )

    df["КлючУчета"] = np.where(
        fio == "Вакансия",
        подразделение + "|" + профессия + "|" + дата_начала + "|" + дата_окончания,
        fio,
    )

    prof_replace = {
        "Электрослесарь (слесарь) дежурный и по ремонту оборудования": "Электрослесарь дежурный по ремонту оборудования",
        "Слесарь ТО": "Слесарь по техническому обслуживанию",
        "Водитель погрузчика": "Машинист погрузчика",
        "Мастер горный": "Горный мастер",
        "Электрогазосварщик, занятый на резке и ручной сварке": "Электрогазосварщик",
        "Комендант общежития": "Комендант",
        "Водитель автомобиля - экспедитор": "Водитель-экспедитор",
        "Слесарь на монтаже промывочных установок": "Слесарь-монтажник промывочного оборудования",
        "Слесарь дежурный по ремонту оборудования": "Слесарь",
        "Машинист колесного бульдозера": "Машинист погрузчика",
        "Машинист буровой установки POC": "Машинист буровой установки D55",
    }

    if "Профессия" in df.columns:
        df["Профессия"] = df["Профессия"].replace(prof_replace)

    df.drop(columns=["КлючУчета"], errors="ignore", inplace=True)

    if "Id" in df.columns:
        df["Id"] = pd.to_numeric(df["Id"], errors="coerce").astype("Int64")

    df = add_resource_shift_columns(df)

    return df


In [20]:
base_path = Path("source")

initial_plan_path = base_path / "Первоначальный план.xlsx"
plan_count_path = base_path / "План_численность.xlsx"
fact_count_path = base_path / "Факт_численность.xlsx"

df_initial_plan_raw = pd.read_excel(initial_plan_path, sheet_name="Лист1")
df_plan_count_raw = pd.read_excel(plan_count_path, sheet_name="Лист1")
df_fact_count_raw = pd.read_excel(fact_count_path, sheet_name="Лист1")

df_initial_plan = transform_initial_plan(df_initial_plan_raw)
df_plan_count = transform_staff_count(df_plan_count_raw)
df_fact_count = transform_staff_count(df_fact_count_raw)

result = transform_registry(df_initial_plan, df_plan_count, df_fact_count)

output_path = "Реестр_обработанный.xlsx"
result.to_excel(output_path, index=False)

print("Файл сохранён:", output_path)
print("Размер итоговой таблицы:", result.shape)
print(result.head())


print("Новые столбцы:", [c for c in result.columns if "Ресурсы_" in c or "t+1" in c or "t+7" in c])


Файл сохранён: Реестр_обработанный.xlsx
Размер итоговой таблицы: (457462, 19)
   Подразделение                         План/Факт                 ЦФО  \
0  Администрация  Ресурсы для выполнения программы  Строительный отдел   
1  Администрация  Ресурсы для выполнения программы  Строительный отдел   
2  Администрация  Ресурсы для выполнения программы  Строительный отдел   
3  Администрация  Ресурсы для выполнения программы  Строительный отдел   
4  Администрация  Ресурсы для выполнения программы  Строительный отдел   

  Профессия                       ФИО Дата начала работы  \
0   Плотник  Ворзуль Максим Сергеевич         2026-02-25   
1   Плотник  Ворзуль Максим Сергеевич         2026-02-25   
2   Плотник  Ворзуль Максим Сергеевич         2026-02-25   
3   Плотник  Ворзуль Максим Сергеевич         2026-02-25   
4   Плотник  Ворзуль Максим Сергеевич         2026-02-25   

  Дата окончания работы       Дата  Id   Год ДеньМесяц    Месяц  Ресурсы_t  \
0            2026-11-25 2026-02-25   1

In [21]:
result

,Подразделение,План/Факт,ЦФО,Профессия,ФИО,Дата начала работы,Дата окончания работы,Дата,Id,Год,ДеньМесяц,Месяц,Ресурсы_t,Ресурсы_t+1,Ресурсы_t+7,Дата_t+1,Дата_t+7,ДеньМесяц_t+1,ДеньМесяц_t+7
0,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-25,1,2026,25.02,февраль,1,1,1,2026-02-24,2026-02-18,24.02,18.02
1,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-26,1,2026,26.02,февраль,1,1,1,2026-02-25,2026-02-19,25.02,19.02
2,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-27,1,2026,27.02,февраль,1,1,1,2026-02-26,2026-02-20,26.02,20.02
3,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-02-28,1,2026,28.02,февраль,1,1,1,2026-02-27,2026-02-21,27.02,21.02
4,Администрация,Ресурсы для выполнения программы,Строительный отдел,Плотник,Ворзуль Максим Сергеевич,2026-02-25,2026-11-25,2026-03-01,1,2026,01.03,март,1,1,1,2026-02-28,2026-02-22,28.02,22.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458450,Дражный,Факт,<NA>,Старший механик,Якубец Виктор Геннадьевич,2026-03-10,2026-10-30,2026-10-26,23420,2026,26.10,октябрь,0,0,0,NaT,NaT,<NA>,<NA>
458451,Дражный,Факт,<NA>,Старший механик,Якубец Виктор Геннадьевич,2026-03-10,2026-10-30,2026-10-27,23420,2026,27.10,октябрь,0,0,0,NaT,NaT,<NA>,<NA>
458452,Дражный,Факт,<NA>,Старший механик,Якубец Виктор Геннадьевич,2026-03-10,2026-10-30,2026-10-28,23420,2026,28.10,октябрь,0,0,0,NaT,NaT,<NA>,<NA>
458453,Дражный,Факт,<NA>,Старший механик,Якубец Виктор Геннадьевич,2026-03-10,2026-10-30,2026-10-29,23420,2026,29.10,октябрь,0,0,0,NaT,NaT,<NA>,<NA>
